In [1]:
import pandas as pd, numpy as np, torch, torchvision, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import sys
sys.path.insert(0, '../')
from util import *
from models.cnn import CNN

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)


# access each scene folder
scenarios = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
val_test_dir = ()


device: NVIDIA A30


# Helper functions

In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    """Load one scenario, run preprocessing, return tensors + shapes."""
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv("../CSV_Files/val.csv").to_numpy()
    test  = pd.read_csv("../CSV_Files/test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(device, seed=0):
    """Fresh CNN with Xavier init, to match MATLAB's Glorot default."""
    torch.manual_seed(seed)
    model = CNN(n_classes=8).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    """MATLAB-style 'shuffle once' then fixed-order iteration."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def run_scenario(scenario_idx, scenario_dir, model_cls, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    # ---- compute metrics setup ----
    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    # ---- TRAINING with timer ----
    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- INFERENCE timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- TEST accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        # ---- new compute fields ----
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }




In [3]:
results = []
for i, sc_dir in enumerate(scenarios, start=1):
    r = run_scenario(i, sc_dir, model_cls=CNN, device=device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 (CNN) ===
  params: 644  breakdown: {'conv': 36, 'bn': 24, 'fc': 584}
  ep   1 | train loss 1.6540 acc 0.419 | val loss 1.4926 acc 0.492
  ep  10 | train loss 0.7664 acc 0.705 | val loss 0.8212 acc 0.740
  ep  20 | train loss 0.6141 acc 0.768 | val loss 0.7598 acc 0.767
  ep  30 | train loss 0.5082 acc 0.814 | val loss 0.6449 acc 0.822
  ep  40 | train loss 0.4773 acc 0.827 | val loss 0.6359 acc 0.831
  ep  50 | train loss 0.4373 acc 0.845 | val loss 0.5834 acc 0.874
  ep  60 | train loss 0.4257 acc 0.849 | val loss 0.5789 acc 0.877
  ep  70 | train loss 0.4046 acc 0.859 | val loss 0.5754 acc 0.885
  TEST: acc=0.8719  P=0.8722  R=0.8719  F1=0.8720
  COMPUTE: train=32.3s  inf=0.000ms/sample  peak_mem=17.4MB

=== Scenario 2 (CNN) ===
  params: 644  breakdown: {'conv': 36, 'bn': 24, 'fc': 584}
  ep   1 | train loss 1.8221 acc 0.364 | val loss 1.6519 acc 0.471
  ep  10 | train loss 0.9754 acc 0.633 | val loss 1.1047 acc 0.608
  ep  20 | train loss 0.8058 acc 0.694 | val los

In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(summary.to_string(index=False))

summary.to_csv("cnn_results_by_scenario.csv", index=False)


 scenario model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1   CNN     7860    0.8719     0.8722  0.8719 0.8720       644      32.26             0.0002         17.4
        2   CNN     3912    0.8169     0.8223  0.8169 0.8151       644      14.78             0.0002         17.4
        3   CNN      778    0.6881     0.7036  0.6881 0.6837       644       4.55             0.0002         17.4
        4   CNN      394    0.5762     0.5928  0.5762 0.5683       644       3.27             0.0002         17.4
        5   CNN      236    0.5494     0.5757  0.5494 0.5563       644       2.87             0.0002         17.4
        6   CNN       75    0.3475     0.3514  0.3475 0.3388       644       2.23             0.0002         17.4


In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  (n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (n_train=7860, acc=0.872)
[[157   0   1  35   0   2   4   1]
 [  2 172   0   5   0  11   0  10]
 [  0   0 192   0   0   5   0   3]
 [ 39   1   0 149   0   0  10   1]
 [  0   0   0   0 200   0   0   0]
 [  0  10   1   0   0 170   0  19]
 [  2   0   0  11   0   0 187   0]
 [  0  19   3   1   0   9   0 168]]

Scenario 2  (n_train=3912, acc=0.817)
[[135   3   3  39   0   3  16   1]
 [  4 163   0   1   0   2   0  30]
 [  0   0 198   0   0   0   0   2]
 [ 59   2   0 120   0   0  18   1]
 [  0   0   0   0 200   0   0   0]
 [  0  11   7   0   1 136   0  45]
 [  1   0   0  16   0   0 183   0]
 [  0  16   5   4   0   3   0 172]]

Scenario 3  (n_train=778, acc=0.688)
[[ 73  16   0  59   0   6  29  17]
 [ 22 122   1   8   0   1   0  46]
 [  0   0 182   1   0   3   0  14]
 [ 49   3   0 103   0   0  35  10]
 [  0   0   0   0 200   0   0   0]
 [  0   9  23   0   2 104   0  62]
 [  3   0   0  36   0   1 159   1]
 [  2  14  18   4   0   3   1 158]]

Scenario 4  (n_train=394, acc=0.576)
[[ 